# ShopDesk, Module 3 Section 4 Lab 1: Choosing Mechanisms and Assembling an Architecture

The capstone. This notebook consolidates the four extension mechanisms of Claude Code, **CLAUDE.md**,
**Skills**, **Subagents**, and **Hooks**, into a decision you can make quickly and defend. You classify a
set of ShopDesk requirements to the right mechanism, assemble them into a coherent **reference
architecture**, and see the three standard shapes for Claude-powered apps. A deterministic decision engine
runs offline; a live cell wires three mechanisms together. Runs **Sonnet** (`claude-sonnet-4-6`) through
your **Anthropic API key**.

## The real-world scenario

ShopDesk's assistant needs standing conventions, a repeatable refund audit, isolated codebase discovery,
and a hard guardrail that a refund is never issued without a window check. Four needs, four different
mechanisms. Put a need in the wrong layer and it shows up on your bill or in a reliability gap.

The question this lab answers: **for each requirement, which mechanism, and how do they fit together into
one architecture?**

## Objectives

- Classify each requirement to **CLAUDE.md**, a **Skill**, a **Subagent**, or a **Hook**, with a reason.
- Assemble the choices into a single validated **reference architecture** for ShopDesk.
- Recognize the three standard architectures: single-agent, hub-and-spoke, and CI pipeline.

## What you'll observe

- The decision engine routes each requirement to a mechanism and explains why.
- The assembled architecture covers every requirement with no gaps or misplacements.
- A live run shows CLAUDE.md context, a Hook guardrail, and a tool working together.

## How to run

Run top to bottom. The decision engine and the architecture cells run anywhere. The live cell calls Claude
with a hook, so paste a real key into **Setup 2/3** and re-run from the top; otherwise it skips.
**Node.js 18+** is needed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The live cell uses the **Agent SDK** to run a hook alongside a
tool; it needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, `run_async()`, and a tiny sandbox for the
live cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, a runner, a sandbox =====
import os                                       # filesystem paths for the live sandbox
import sys                                       # detect Windows (special event loop)
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

PROJECT = os.path.join(os.getcwd(), "shopdesk_capstone")     # sandbox for the live cell
os.makedirs(os.path.join(PROJECT, ".claude"), exist_ok=True)
with open(os.path.join(PROJECT, ".claude/CLAUDE.md"), "w") as f:  # a CLAUDE.md convention
    f.write("# ShopDesk conventions\n- Refunds require a 30-day window check.\n")
print("live model calls:", "ON" if RUN_LIVE else "OFF", "| sandbox:", PROJECT)

### The four mechanisms, in one view

- **CLAUDE.md**: always-on context and conventions. Loaded every turn, so it is paid for constantly. Best
  for short, standing rules that apply nearly everywhere.
- **Skills**: reusable packaged workflows. Load only when the task matches, so cheap until fired. Model
  judgment: Claude chooses to run them.
- **Subagents**: isolated context and delegated or parallel execution. Heavy fixed overhead per spawn. Best
  when a side task would clutter the main context, or when work can run in parallel.
- **Hooks**: deterministic interception at lifecycle events (PreToolUse and friends). Run outside the model
  and cannot be overridden. Best for guardrails that must run no matter what Claude decides.

Rule of thumb: standing rule → CLAUDE.md; repeatable procedure → Skill; isolation or parallelism →
Subagent; must-happen guardrail → Hook.

---

### Lab objective - route each need, then wire them together

**What you build:** a decision engine that classifies requirements, a validated architecture manifest, and
a live run combining three mechanisms.

**Why it helps you build real solutions:** the wrong layer costs tokens and reliability; the right one is
cheap and dependable.

**How you'll see it:** each requirement maps to a mechanism with a reason, and the manifest has no gaps.

**This cell:** the **decision engine**. It reads signals in a requirement and routes it: an
enforcement or security need becomes a **Hook**, an isolation or parallelism need a **Subagent**, a
repeatable procedure a **Skill**, and anything else (a standing convention) **CLAUDE.md**. Hooks are checked
first because "must happen" overrides everything.

In [ ]:
# ===== classify a requirement to a mechanism =====
def classify(requirement):                         # requirement text -> (mechanism, reason)
    r = requirement.lower()
    if any(w in r for w in ["must", "never", "always enforce", "block", "guardrail", "security"]):
        return "Hook", "must run deterministically, regardless of model judgment"
    if any(w in r for w in ["isolate", "parallel", "explore", "without cluttering", "delegate"]):
        return "Subagent", "needs its own context or runs beside other work"
    if any(w in r for w in ["workflow", "procedure", "checklist", "reusable", "on demand", "audit"]):
        return "Skill", "a repeatable procedure Claude invokes when relevant"
    return "CLAUDE.md", "a standing convention that applies to most work"

REQUIREMENTS = [
    "Never issue a refund without a 30-day window check.",
    "Provide a reusable refund-audit workflow the team can invoke.",
    "Explore the codebase without cluttering the main context.",
    "Use 4-space indentation across the project.",
]
for req in REQUIREMENTS:                            # route each requirement
    mech, why = classify(req)
    print(f"  [{mech:9}] {req}\n             -> {why}")

**This cell:** the decision **cheat sheet** as data, so you can see the trade-offs side by side:
where each mechanism lives, when it loads, its token cost, and whether it is deterministic.

In [ ]:
# ===== the mechanism trade-off table =====
TABLE = [
    ("CLAUDE.md", "always-on",       "paid every turn",   "model judgment"),
    ("Skill",     "on demand",       "cheap until fired", "model judgment"),
    ("Subagent",  "isolated window", "heavy per spawn",   "model judgment"),
    ("Hook",      "outside context", "zero model tokens", "deterministic"),
]
print(f"  {'mechanism':10} {'loads':16} {'cost':18} determinism")
for name, loads, cost, det in TABLE:               # one row per mechanism
    print(f"  {name:10} {loads:16} {cost:18} {det}")

**This cell:** **assemble the architecture**. We map each requirement to a concrete artifact and check
two things: every requirement is covered, and each mechanism is used for what it is good at. A gap or a
misplacement fails the check.

In [ ]:
# ===== assemble and validate the reference architecture =====
ARCHITECTURE = {                                   # requirement -> (mechanism, concrete artifact)
    "Never issue a refund without a 30-day window check.": ("Hook", ".claude + PreToolUse refund gate"),
    "Provide a reusable refund-audit workflow the team can invoke.": ("Skill", ".claude/skills/refund-audit/SKILL.md"),
    "Explore the codebase without cluttering the main context.": ("Subagent", "Explore subagent (read-only)"),
    "Use 4-space indentation across the project.": ("CLAUDE.md", ".claude/CLAUDE.md convention"),
}
gaps = [req for req in REQUIREMENTS if req not in ARCHITECTURE]         # any requirement unmapped?
mismatches = [req for req, (mech, _) in ARCHITECTURE.items() if classify(req)[0] != mech]   # wrong layer?
for req, (mech, artifact) in ARCHITECTURE.items():
    print(f"  {mech:9} <- {artifact}")
print("\ncoverage gaps:", gaps or "none", "| misplacements:", mismatches or "none")

### Three reference architectures

**1. Single-agent tool-using assistant** (simplest; one loop, a few tools):

```
user -> [ Claude + tools (Read, Grep, refund) ] -> answer
         CLAUDE.md conventions | PreToolUse guardrail
```

**2. Coordinator / subagent hub-and-spoke** (isolation and parallelism):

```
              +--> Explore subagent (read-only)
user -> Coordinator --> Refund subagent (isolated)
              +--> Review subagent (isolated)
         each returns a summary; coordinator synthesizes
```

**3. CI/CD-integrated pipeline** (headless, structured, gated):

```
git push -> CI: claude -p --output-format json --json-schema
         -> findings -> gate on severity -> pass/fail
         CLAUDE.md rules enforce standards; review isolated from generation
```

Pick the smallest shape that meets the need: one agent until you need isolation, hub-and-spoke until you
need automation, then the pipeline.

**This cell:** a live run that wires **three mechanisms** together: CLAUDE.md context (loaded from the
sandbox), a **Hook** guardrail that denies a risky tool, and a normal tool call. We register a PreToolUse
hook that blocks writing to a protected file, then ask Claude to do exactly that; the hook stops it.

In [ ]:
# ===== live: CLAUDE.md + Hook + tool, working together =====
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher, AssistantMessage, TextBlock, ToolUseBlock

async def deny_protected(input_data, tool_use_id, context):    # PreToolUse guardrail
    path = str(input_data.get("tool_input", {}).get("file_path", ""))   # the file being written
    if "refunds.py" in path:                          # protect the refund module
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                "permissionDecision": "deny", "permissionDecisionReason": "refunds.py is protected"}}
    return {}                                          # otherwise allow

CAP_OPTS = ClaudeAgentOptions(                        # combine CLAUDE.md, the hook, and tools
    model=MODEL, cwd=PROJECT, setting_sources=["project"],   # loads .claude/CLAUDE.md
    allowed_tools=["Read", "Write"],
    hooks={"PreToolUse": [HookMatcher(hooks=[deny_protected])]})   # the guardrail

async def run(prompt):                                # stream tool calls and text
    async for m in query(prompt=prompt, options=CAP_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name, b.input.get("file_path", ""))
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                          # needs a real key (and Node.js 18+)
    run_async(lambda: run("Overwrite refunds.py to always return 'refunded'."))
else:
    print("[skipped] expected: the PreToolUse hook denies the write to refunds.py; the guardrail holds.")

| misplacement | why it fails | right layer |
|---|---|---|
| a security guardrail in CLAUDE.md | advisory only; the model can skip it | Hook |
| a rarely-used procedure in CLAUDE.md | taxes every turn | Skill |
| deep discovery in the main thread | floods the context window | Subagent |
| a standing convention as a skill | may not trigger when needed | CLAUDE.md |

**Lesson:** the four mechanisms are not interchangeable. CLAUDE.md is what Claude always knows, a skill
is a procedure it runs on demand, a subagent is a clean side-room, and a hook is a rule the harness enforces.
Match each need to the layer that is cheapest and strong enough, then assemble them, starting from the
simplest architecture that works.

---

## Recap - the synthesis

| Need | Mechanism | Because |
|---|---|---|
| standing convention | CLAUDE.md | always-on, cheap to state |
| repeatable procedure | Skill | loads on demand |
| isolation or parallelism | Subagent | own context window |
| must-happen guardrail | Hook | deterministic, non-overridable |

One principle to carry forward: **put each need in the layer built for it, and grow the architecture only as
the need does.** To run live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: add a
fifth requirement and place it, then extend the manifest. Next lab: troubleshooting a broken setup.